In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

In [13]:
data_path = 'healthcare.xlsx'
df = pd.read_excel(data_path)

In [14]:
df_cleaned = df[df['Age'] >= 0].copy()
df_cleaned['ScheduledDay'] = pd.to_datetime(df_cleaned['ScheduledDay'])
df_cleaned['AppointmentDay'] = pd.to_datetime(df_cleaned['AppointmentDay'])
df_cleaned['ScheduledDate'] = df_cleaned['ScheduledDay'].dt.normalize()
df_cleaned['AppointmentDate'] = df_cleaned['AppointmentDay'].dt.normalize()
df_cleaned['LeadTime'] = (df_cleaned['AppointmentDate'] - df_cleaned['ScheduledDate']).dt.days
df_cleaned = df_cleaned[df_cleaned['LeadTime'] >= 0].copy()
df_cleaned['Appointment_Weekday'] = df_cleaned['AppointmentDate'].dt.day_name()
df_cleaned['NoShow_numeric'] = df_cleaned['No-show'].apply(lambda x: 1 if x == 'Yes' else 0)

In [15]:
features_to_use = ['Gender', 'Age', 'Scholarship', 'Hipertension', 'Diabetes', 
                   'Alcoholism', 'Handcap', 'SMS_received', 'LeadTime', 'Appointment_Weekday']
X = df_cleaned[features_to_use].copy()
y = df_cleaned['NoShow_numeric']
X['Gender'] = X['Gender'].map({'F': 0, 'M': 1})
X = pd.get_dummies(X, columns=['Appointment_Weekday'], drop_first=True, dtype=int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [16]:
dt_model = DecisionTreeClassifier(max_depth=6, random_state=42, class_weight='balanced')
dt_model.fit(X_train, y_train)
y_pred = dt_model.predict(X_test)
y_pred_proba = dt_model.predict_proba(X_test)[:, 1]
print("\n================ MODEL PERFORMANCE EVALUATION ================")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}\n")


================ MODEL PERFORMANCE EVALUATION ================
Accuracy Score: 0.5825
ROC-AUC Score: 0.7217



In [17]:
X_full = df_cleaned[features_to_use].copy()
X_full['Gender'] = X_full['Gender'].map({'F': 0, 'M': 1})
X_full = pd.get_dummies(X_full, columns=['Appointment_Weekday'], drop_first=True, dtype=int)
X_full = X_full.reindex(columns=X_train.columns, fill_value=0)
df_cleaned['Predicted_NoShow'] = dt_model.predict(X_full)
df_cleaned['NoShow_Probability'] = dt_model.predict_proba(X_full)[:, 1]
def group_lead_time(days):
    if days == 0: return 'Same Day (0)'
    elif days <= 3: return '1-3 Days'
    elif days <= 7: return '4-7 Days'
    elif days <= 14: return '8-14 Days'
    elif days <= 30: return '15-30 Days'
    else: return '31+ Days'
def group_age(age):
    if age <= 12: return 'Child (0-12)'
    elif age <= 19: return 'Teen (13-19)'
    elif age <= 35: return 'Young Adult (20-35)'
    elif age <= 60: return 'Adult (36-60)'
    else: return 'Senior (61+)'
df_cleaned['LeadTime_Group'] = df_cleaned['LeadTime'].apply(group_lead_time)
df_cleaned['Age_Group'] = df_cleaned['Age'].apply(group_age)
output_filename = 'cleaned_healthcare_predictions.csv'
df_cleaned.to_csv(output_filename, index=False)
print(f"\nSuccess! Fixed file exported as '{output_filename}' without data type conflicts.")


Success! Fixed file exported as 'cleaned_healthcare_predictions.csv' without data type conflicts.
